# 02.1 · xLSTM (sLSTM / mLSTM)

Part of the study *Multi-Label 12-Lead ECG Classification on the PTB-XL Dataset* ([arXiv:2609.12803](https://arxiv.org/abs/2609.12803)). Run order, folder layout, and results: [project README](../README.md).

> GPU: L4. `batch_size=32`.

Shared protocol (`ecg_common.py`): official PTB-XL folds 1–8 / 9 / 10, checkpoint = validation macro-AUPRC, val F1-max thresholds, reported metrics = test fold 10. Outputs: `experiments/<model>/seed_<seed>/` and `results/<canonical>/seed_<seed>/`.
Reads `processed/sr100` from notebook 00.


In [ ]:
!pip install -q openpyxl scikit-learn pandas numpy matplotlib seaborn torch tqdm


In [ ]:
from google.colab import drive
from pathlib import Path
import sys

drive.mount('/content/drive')

SAMPLING_RATE = 100
DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_DIR = DRIVE_ROOT / 'ECG AI'
PROCESSED_DIR = PROJECT_DIR / 'processed' / f'sr{SAMPLING_RATE}'
EXPERIMENTS_DIR = PROJECT_DIR / 'experiments'
RESULTS_DIR = PROJECT_DIR / 'results'
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))
from ecg_common import load_processed_manifest

_manifest = load_processed_manifest(PROCESSED_DIR)
print('processed:', PROCESSED_DIR)
print('dataset_version:', _manifest.get('dataset_version') or _manifest.get('raw_root_name'))
print('split:', _manifest.get('split_by'), _manifest.get('train_folds'), _manifest.get('val_folds'), _manifest.get('test_folds'))


## Shared protocol

`notebooks/ecg_common.py` is imported. Checkpoint / early stop: **validation macro-AUPRC**. Architecture stays in this notebook; splits, metrics, thresholds, and the Trainer live in the shared module.


In [ ]:
from __future__ import annotations

import sys

import numpy as np
import pandas as pd
import torch
from torch import nn

sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))
from ecg_common import (
    CLASSES,
    EARLY_STOPPING_PATIENCE,
    EVALUATE_TEST as DEFAULT_EVALUATE_TEST,
    ExperimentConfig,
    MAX_EPOCHS,
    PRIMARY_MONITOR,
    SEEDS,
    build_trainer,
    finalize_run_config,
)

EVALUATE_TEST = DEFAULT_EVALUATE_TEST  # True: score test fold 10 with val-tuned thresholds
ACTIVE_SEEDS = [42]  # paper: ACTIVE_SEEDS = SEEDS
print('monitor=', PRIMARY_MONITOR, '| epochs=', MAX_EPOCHS, '| patience=', EARLY_STOPPING_PATIENCE)
print('ACTIVE_SEEDS', ACTIVE_SEEDS, '| EVALUATE_TEST', EVALUATE_TEST)


## Model

Keep architecture hyperparameters (`d_model`, `depth`, `slstm_at`, `bidirectional`, …) in the `model_factory` below **and** in `experiment.extra` together; changing only one side does not update the model.


In [ ]:
# MODEL_ARCHITECTURE
from typing import Tuple

class CausalConv1d(nn.Module):
    def __init__(self, channels: int, kernel_size: int) -> None:
        super().__init__()
        self.pad = kernel_size - 1
        self.conv = nn.Conv1d(
            channels,
            channels,
            kernel_size=kernel_size,
            groups=channels,
            padding=self.pad,
            bias=True,
        )
        self.act = nn.SiLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.transpose(1, 2)
        x = self.conv(x)
        if self.pad:
            x = x[:, :, :-self.pad]
        return self.act(x.transpose(1, 2))


class FeedForward(nn.Module):
    def __init__(self, d_model: int, proj_factor: float = 1.3, dropout: float = 0.0) -> None:
        super().__init__()
        hidden = max(1, int(round(d_model * proj_factor)))
        self.net = nn.Sequential(
            nn.Linear(d_model, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class sLSTM(nn.Module):
    """Scalar LSTM with exponential gating, stabilizer, and head-wise memory mixing."""

    def __init__(self, d_model: int, num_heads: int) -> None:
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError(f'd_model={d_model} must be divisible by num_heads={num_heads}')
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.wx = nn.Linear(d_model, 4 * d_model)
        self.R = nn.Parameter(torch.empty(4, num_heads, self.head_dim, self.head_dim))
        nn.init.xavier_uniform_(self.R)
        nn.init.zeros_(self.wx.bias)
        self.wx.bias.data[d_model:2 * d_model].fill_(1.0)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, steps, _ = x.shape
        heads, dim = self.num_heads, self.head_dim
        i_x, f_x, z_x, o_x = self.wx(x).chunk(4, dim=-1)
        hidden = x.new_zeros(batch, heads, dim)
        cell = x.new_zeros(batch, heads, dim)
        normalizer = x.new_zeros(batch, heads, dim)
        stabilizer = x.new_zeros(batch, heads, dim)
        outputs = []
        for t in range(steps):
            rec = torch.einsum('bhd,nhde->nbhe', hidden, self.R)
            i_pre = i_x[:, t].view(batch, heads, dim) + rec[0]
            f_pre = f_x[:, t].view(batch, heads, dim) + rec[1]
            z_t = torch.tanh(z_x[:, t].view(batch, heads, dim) + rec[2])
            o_t = torch.sigmoid(o_x[:, t].view(batch, heads, dim) + rec[3])
            m_t = torch.maximum(f_pre + stabilizer, i_pre)
            i_gate = torch.exp(i_pre - m_t)
            f_gate = torch.exp(f_pre + stabilizer - m_t)
            cell = f_gate * cell + i_gate * z_t
            normalizer = f_gate * normalizer + i_gate
            hidden = o_t * (cell / normalizer.clamp_min(1e-6))
            stabilizer = m_t
            outputs.append(hidden)
        y = torch.stack(outputs, dim=1).reshape(batch, steps, -1)
        return self.out_proj(y)


class mLSTM(nn.Module):
    """Matrix LSTM with covariance memory, exponential gating, and stabilizer."""

    def __init__(self, d_model: int, num_heads: int) -> None:
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError(f'd_model={d_model} must be divisible by num_heads={num_heads}')
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.igate = nn.Linear(d_model, num_heads)
        self.fgate = nn.Linear(d_model, num_heads)
        self.ogate = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        nn.init.zeros_(self.igate.bias)
        nn.init.ones_(self.fgate.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, steps, _ = x.shape
        heads, dim = self.num_heads, self.head_dim
        q, k, v = self.qkv(x).view(batch, steps, 3, heads, dim).unbind(dim=2)
        k = k * self.scale
        i_pre = self.igate(x)
        f_pre = self.fgate(x)
        o_t = torch.sigmoid(self.ogate(x)).view(batch, steps, heads, dim)
        memory = x.new_zeros(batch, heads, dim, dim)
        normalizer = x.new_zeros(batch, heads, dim)
        stabilizer = x.new_zeros(batch, heads)
        outputs = []
        for t in range(steps):
            qt, kt, vt = q[:, t], k[:, t], v[:, t]
            ip, fp = i_pre[:, t], f_pre[:, t]
            m_t = torch.maximum(fp + stabilizer, ip)
            i_gate = torch.exp(ip - m_t)
            f_gate = torch.exp(fp + stabilizer - m_t)
            memory = (
                f_gate[:, :, None, None] * memory
                + i_gate[:, :, None, None] * torch.einsum('bhd,bhe->bhde', vt, kt)
            )
            normalizer = f_gate[:, :, None] * normalizer + i_gate[:, :, None] * kt
            num = torch.einsum('bhde,bhe->bhd', memory, qt)
            den = torch.einsum('bhd,bhd->bh', normalizer, qt).abs().clamp_min(1.0).unsqueeze(-1)
            outputs.append(o_t[:, t] * (num / den))
            stabilizer = m_t
        y = torch.stack(outputs, dim=1).reshape(batch, steps, -1)
        return self.out_proj(y)


def _make_sequence_cell(kind: str, d_model: int, num_heads: int) -> nn.Module:
    if kind == 'slstm':
        return sLSTM(d_model, num_heads)
    if kind == 'mlstm':
        return mLSTM(d_model, num_heads)
    raise ValueError(f'Unknown xLSTM cell kind: {kind}')


class XLSTMBlock(nn.Module):
    def __init__(
        self,
        d_model: int,
        num_heads: int,
        conv_kernel: int,
        kind: str,
        dropout: float,
        bidirectional: bool,
        proj_factor: float = 1.3,
    ) -> None:
        super().__init__()
        self.bidirectional = bidirectional
        self.norm = nn.LayerNorm(d_model)
        self.conv = CausalConv1d(d_model, conv_kernel)
        self.cell = _make_sequence_cell(kind, d_model, num_heads)
        if bidirectional:
            self.conv_bwd = CausalConv1d(d_model, conv_kernel)
            self.cell_bwd = _make_sequence_cell(kind, d_model, num_heads)
        self.drop = nn.Dropout(dropout)
        self.ffn = None
        if kind == 'slstm':
            self.ffn_norm = nn.LayerNorm(d_model)
            self.ffn = FeedForward(d_model, proj_factor=proj_factor, dropout=dropout)

    def _unidir(self, x: torch.Tensor, conv: nn.Module, cell: nn.Module) -> torch.Tensor:
        return cell(conv(x))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.norm(x)
        if self.bidirectional:
            forward_out = self._unidir(h, self.conv, self.cell)
            backward_out = torch.flip(
                self._unidir(torch.flip(h, dims=[1]), self.conv_bwd, self.cell_bwd),
                dims=[1],
            )
            y = 0.5 * (forward_out + backward_out)
        else:
            y = self._unidir(h, self.conv, self.cell)
        x = x + self.drop(y)
        if self.ffn is not None:
            x = x + self.drop(self.ffn(self.ffn_norm(x)))
        return x


class XLSTMClassifier(nn.Module):
    def __init__(
        self,
        in_channels: int = 12,
        num_classes: int = 5,
        d_model: int = 64,
        depth: int = 2,
        num_heads: int = 4,
        dropout: float = 0.2,
        slstm_at: Tuple[int, ...] = (1,),
        bidirectional: bool = False,
        conv_kernel: int = 4,
    ) -> None:
        super().__init__()
        slstm_layers = set(slstm_at)
        self.input_proj = nn.Conv1d(in_channels, d_model, kernel_size=5, padding=2)
        self.blocks = nn.ModuleList(
            [
                XLSTMBlock(
                    d_model=d_model,
                    num_heads=num_heads,
                    conv_kernel=conv_kernel,
                    kind='slstm' if idx in slstm_layers else 'mlstm',
                    dropout=dropout,
                    bidirectional=bidirectional,
                )
                for idx in range(depth)
            ]
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_proj(x).transpose(1, 2)
        for block in self.blocks:
            x = block(x)
        x = x.mean(dim=1)
        x = self.dropout(x)
        return self.classifier(x)


_smoke = XLSTMClassifier()
_x = torch.randn(2, 12, 1000)
_y = _smoke(_x)
assert _y.shape == (2, 5), _y.shape
print(
    'xlstm OK |',
    'out', tuple(_y.shape),
    '| params', sum(p.numel() for p in _smoke.parameters()),
)
del _smoke, _x, _y


## Training hyperparameters

Hyperparameters are defined in `ExperimentConfig(...)` in the next cell.

xLSTM: `batch_size=32`, `learning_rate=5e-4`. Shared protocol: `MAX_EPOCHS=50`, `EARLY_STOPPING_PATIENCE=7`, checkpoint = validation **macro-AUPRC**. `ACTIVE_SEEDS = [42]` is a single seed; for the paper use `ACTIVE_SEEDS = SEEDS` (`[42, 52, 62]`).
Checkpoints and thresholds are fit on val fold 9; reported performance analysis uses test fold 10.


In [ ]:
experiment = ExperimentConfig(
    model_name='xlstm',
    processed_dir=str(PROCESSED_DIR),
    output_root=str(EXPERIMENTS_DIR),
    learning_rate=5e-4,
    weight_decay=1e-4,
    epochs=MAX_EPOCHS,
    batch_size=32,
    sampling_rate=SAMPLING_RATE,
    seed=42,
    patience=EARLY_STOPPING_PATIENCE,
    monitor=PRIMARY_MONITOR,
    evaluate_test=EVALUATE_TEST,
    results_root=str(RESULTS_DIR),
    extra={
        'd_model': 64,
        'depth': 2,
        'num_heads': 4,
        'dropout': 0.2,
        'slstm_at': [1],
        'bidirectional': False,
        'conv_kernel': 4,
    },
)

def model_factory(num_classes: int):
    return XLSTMClassifier(
        in_channels=12,
        num_classes=num_classes,
        d_model=64,
        depth=2,
        num_heads=4,
        dropout=0.2,
        slstm_at=(1,),
        bidirectional=False,
        conv_kernel=4,
    )

print('config:', experiment.model_name, '| lr', experiment.learning_rate, '| batch', experiment.batch_size, '| epochs', experiment.epochs)
print('X_train', tuple(np.load(Path(experiment.processed_dir) / 'X_train.npy', mmap_mode='r').shape))


In [ ]:
for seed in ACTIVE_SEEDS:
    experiment.seed = int(seed)
    trainer, data_module = build_trainer(experiment, model_factory=model_factory)
    history_df = trainer.fit(
        data_module.train_dataloader(),
        data_module.val_dataloader(),
        epochs=experiment.epochs,
    )
    protocol = trainer.evaluate_with_protocol(
        data_module.val_dataloader(),
        test_loader=data_module.test_dataloader(),
        evaluate_test=EVALUATE_TEST,
        results_root=RESULTS_DIR,
    )
    finalize_run_config(
        trainer,
        extra={'dataset_version': data_module.manifest.get('dataset_version') or data_module.manifest.get('raw_root_name')},
    )
    print('best val macro-AUPRC', protocol.get('best_validation_macro_auprc'), 'epoch', protocol.get('best_epoch'))

history_df.tail()


## Resume from last.pt

If the `fit` cell above was interrupted (Colab disconnect, OOM, runtime reset), run this cell. It loads `last.pt` (weights + optimizer) and writes remaining epochs into the same run folder.

After a kernel restart, skip the training cell: re-run the install, Drive, protocol, model, and `ExperimentConfig` cells, then this cell.
If `trainer` is missing, this cell rebuilds it with `build_trainer`. Do not point it at another model's `last.pt`.


In [ ]:
import json

RESUME_RUN_DIR = None  # None: experiments/xlstm/seed_<seed>

if 'trainer' not in globals() or 'data_module' not in globals():
    trainer, data_module = build_trainer(experiment, model_factory=model_factory)

_resume_dir = Path(RESUME_RUN_DIR) if RESUME_RUN_DIR else (
    Path(experiment.output_root) / experiment.model_name / f'seed_{int(experiment.seed)}'
)
_cfg_path = _resume_dir / 'config.json'
if _cfg_path.exists():
    _cfg = json.loads(_cfg_path.read_text(encoding='utf-8'))
    saved = str(_cfg.get('model_name') or '')
    if saved and saved != experiment.model_name:
        raise ValueError(
            f'Refuse resume: {_resume_dir} belongs to {saved}, this notebook is {experiment.model_name}.'
        )
if not (_resume_dir / 'checkpoints' / 'last.pt').exists():
    raise FileNotFoundError(f'last.pt not found: {_resume_dir / "checkpoints" / "last.pt"}')

history_df = trainer.fit(
    data_module.train_dataloader(),
    data_module.val_dataloader(),
    epochs=experiment.epochs,
    resume_from_last=True,
    resume_run_dir=str(_resume_dir),
)
protocol = trainer.evaluate_with_protocol(
    data_module.val_dataloader(),
    test_loader=data_module.test_dataloader(),
    evaluate_test=EVALUATE_TEST,
    results_root=RESULTS_DIR,
)
history_df.tail()


## Performance analysis

This section is **standalone**: after a runtime restart it can be run on its own, without the install, import, or training cells above.

Loads `experiments/<model>/seed_42/checkpoints/best.pt`. Thresholds are fit on val fold 9 (F1-max). Reported metrics, 2×2 confusion matrices, and prediction CSVs use **test fold 10**.
Files: `experiments/.../predictions/val_predictions.csv` and `test_predictions.csv`; copy under `results/<canonical>/seed_42/`.


In [ ]:
# ANALYZE_FROM_BEST_PT
# This cell runs standalone. Cells above (pip / imports / training) are not required.
import sys
import subprocess
from pathlib import Path
from typing import Tuple  # noqa: F401

subprocess.check_call(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'openpyxl',
        'scikit-learn',
        'pandas',
        'numpy',
        'matplotlib',
        'seaborn',
        'torch',
        'tqdm',
    ]
)
try:
    from google.colab import drive

    drive.mount('/content/drive')
except Exception:
    pass

PROJECT_DIR = Path('/content/drive/MyDrive') / 'ECG AI'
sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))
from ecg_common import (
    evaluate_and_display_best_pt,
    exec_notebook_cells,
    extra_from_run_config,
    load_json,
    seed_run_dir,
)

EVALUATE_TEST = True  # always score test fold 10 from best.pt
ANALYSIS_SEED = 42
MODEL_NAME = 'xlstm'
NOTEBOOK_PATH = PROJECT_DIR / 'notebooks' / '02.1_xlstm.ipynb'
run_dir = seed_run_dir(PROJECT_DIR / 'experiments', MODEL_NAME, ANALYSIS_SEED)
ckpt = run_dir / 'checkpoints' / 'best.pt'
if not ckpt.exists():
    raise FileNotFoundError(f'best.pt not found: {ckpt}')

cfg = load_json(run_dir / 'config.json') if (run_dir / 'config.json').exists() else {}
extra = extra_from_run_config(cfg)

import torch
from torch import nn

exec_notebook_cells(NOTEBOOK_PATH, '# MODEL_ARCHITECTURE', globals())

def model_factory(num_classes: int):
    slstm_at = extra.get('slstm_at', [1])
    return XLSTMClassifier(
        in_channels=12,
        num_classes=num_classes,
        d_model=int(extra.get('d_model', 64)),
        depth=int(extra.get('depth', 2)),
        num_heads=int(extra.get('num_heads', 4)),
        dropout=float(extra.get('dropout', 0.2)),
        slstm_at=tuple(slstm_at),
        bidirectional=bool(extra.get('bidirectional', False)),
        conv_kernel=int(extra.get('conv_kernel', 4)),
    )

evaluate_and_display_best_pt(
    model_factory,
    run_dir,
    processed_dir=PROJECT_DIR / 'processed' / 'sr100',
    results_root=PROJECT_DIR / 'results',
    evaluate_test=True,
)
